In [1]:
# On my computer evaluation

import numpy as np
import time
import tracemalloc
import tensorflow as tf
import glob
import os

# Load test data once
data = np.load("processed_mit_bih_arrhythmia_dataset.npz")
Xe_te, Xr_te, y_te = data['Xe_te'], data['Xr_te'], data['y_te']

# Benchmark all models
models = glob.glob("ecg_classifier_quantized_models/*.tflite")

with open("benchmark_results.txt", "w") as f:
    for model_path in sorted(models):
        name = os.path.basename(model_path)
        print(f"Testing: {name}")
        
        # Load model
        interpreter = tf.lite.Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        inp, out = interpreter.get_input_details(), interpreter.get_output_details()
        
        # Warmup
        for _ in range(10):
            interpreter.set_tensor(inp[0]['index'], Xr_te[0:1].astype(np.float32))
            interpreter.set_tensor(inp[1]['index'], Xe_te[0:1].astype(np.float32))
            interpreter.invoke()
        
        # Measure
        times = []
        correct = 0
        tracemalloc.start()
        
        for i in range(len(Xe_te)):
            start = time.perf_counter()
            interpreter.set_tensor(inp[0]['index'], Xr_te[i:i+1].astype(np.float32))
            interpreter.set_tensor(inp[1]['index'], Xe_te[i:i+1].astype(np.float32))
            interpreter.invoke()
            pred = np.argmax(interpreter.get_tensor(out[0]['index'])[0])
            times.append((time.perf_counter() - start) * 1000)
            correct += (pred == y_te[i])
        
        peak = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
        
        # Write results
        f.write(f"Model: {name}\n")
        f.write(f" Samples: {len(times)}\n")
        f.write(f" Time: {np.mean(times):.2f}±{np.std(times):.2f} ms\n")
        f.write(f" FPS: {1000/np.mean(times):.1f}\n")
        f.write(f" Accuracy: {correct/len(y_te)*100:.2f}%\n")
        f.write(f" Memory: {peak/1024:.2f} KB\n\n")
        
        # Console output
        print(f"  ✓ {correct/len(y_te)*100:.2f}% | {np.mean(times):.2f}ms | {peak/1024:.1f}KB\n")

print("Done! Results saved to benchmark_results.txt")

Testing: conv1d_2d_t_model_float16.tflite
  ✓ 98.66% | 0.13ms | 847.3KB

Testing: conv1d_2d_t_model_int16.tflite
  ✓ 98.66% | 0.16ms | 829.2KB

Testing: conv1d_2d_t_model_int8.tflite
  ✓ 98.66% | 0.12ms | 830.0KB

Testing: conv1d_t_model_float16.tflite
  ✓ 98.31% | 0.05ms | 829.2KB

Testing: conv1d_t_model_int16.tflite
  ✓ 98.31% | 0.05ms | 829.2KB

Testing: conv1d_t_model_int8.tflite
  ✓ 98.34% | 0.05ms | 829.4KB

Testing: pruned_conv1d_2d_t_sp30_float16.tflite
  ✓ 98.80% | 0.16ms | 829.7KB

Testing: pruned_conv1d_2d_t_sp30_int16.tflite
  ✓ 98.79% | 0.16ms | 829.2KB

Testing: pruned_conv1d_2d_t_sp30_int8.tflite
  ✓ 98.81% | 0.12ms | 829.2KB

Testing: pruned_conv1d_2d_t_sp40_float16.tflite
  ✓ 98.69% | 0.16ms | 829.8KB

Testing: pruned_conv1d_2d_t_sp40_int16.tflite
  ✓ 98.68% | 0.16ms | 829.1KB

Testing: pruned_conv1d_2d_t_sp40_int8.tflite
  ✓ 98.71% | 0.11ms | 829.8KB

Testing: pruned_conv1d_2d_t_sp50_float16.tflite
  ✓ 98.55% | 0.16ms | 829.2KB

Testing: pruned_conv1d_2d_t_sp50_int16

In [ ]:
# On Rasbperry PI evaluation
# Use tflite_runtime instead of full tensorflow


import numpy as np
import time
import tracemalloc
import glob
import os
import platform

# Use tflite_runtime instead of full tensorflow
try:
    from tflite_runtime.interpreter import Interpreter
    print("Using tflite_runtime (lightweight)")
except ImportError:
    import tensorflow as tf
    Interpreter = tf.lite.Interpreter
    print("Using full TensorFlow")

# Load test data once
data = np.load("processed_mit_bih_arrhythmia_dataset.npz")
Xe_te, Xr_te, y_te = data['Xe_te'], data['Xr_te'], data['y_te']

# Benchmark all models
models = glob.glob("ecg_classifier_quantized_models/*.tflite")

with open("benchmark_results.txt", "w") as f:
    for model_path in sorted(models):
        name = os.path.basename(model_path)
        print(f"Testing: {name}")
        
        # Load model
        interpreter = Interpreter(model_path=model_path)
        interpreter.allocate_tensors()
        inp, out = interpreter.get_input_details(), interpreter.get_output_details()
        
        # Warmup
        for _ in range(10):
            interpreter.set_tensor(inp[0]['index'], Xr_te[0:1].astype(np.float32))
            interpreter.set_tensor(inp[1]['index'], Xe_te[0:1].astype(np.float32))
            interpreter.invoke()
        
        # Measure
        times = []
        correct = 0
        tracemalloc.start()
        
        for i in range(len(Xe_te)):
            start = time.perf_counter()
            interpreter.set_tensor(inp[0]['index'], Xr_te[i:i+1].astype(np.float32))
            interpreter.set_tensor(inp[1]['index'], Xe_te[i:i+1].astype(np.float32))
            interpreter.invoke()
            pred = np.argmax(interpreter.get_tensor(out[0]['index'])[0])
            times.append((time.perf_counter() - start) * 1000)
            correct += (pred == y_te[i])
        
        peak = tracemalloc.get_traced_memory()[1]
        tracemalloc.stop()
        
        # Write results
        f.write(f"Model: {name}\n")
        f.write(f" Samples: {len(times)}\n")
        f.write(f" Time: {np.mean(times):.2f}±{np.std(times):.2f} ms\n")
        f.write(f" FPS: {1000/np.mean(times):.1f}\n")
        f.write(f" Accuracy: {correct/len(y_te)*100:.2f}%\n")
        f.write(f" Memory: {peak/1024:.2f} KB\n\n")
        
        # Console output
        print(f"{correct/len(y_te)*100:.2f}% | {np.mean(times):.2f}ms | {peak/1024:.1f}KB\n")

print("Done! Results saved to benchmark_results.txt")